# Laboratorio 08.2 — Router de herramientas

Implementaremos un router que asigna solicitudes a funciones permitidas, valida argumentos con Pydantic y mide exactitud de selección por herramienta.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Separar clasificación de intención de ejecución de herramientas.
- Implementar una línea base de enrutamiento y compararla con etiquetas esperadas.
- Validar argumentos antes de despachar una llamada.
- Medir accuracy de selección, validez de argumentos y tasa de fallo.

## 2. Conceptos

Un router asigna una solicitud a una ruta, herramienta o respuesta directa. La selección puede basarse en reglas, un clasificador o una propuesta LLM; la ejecución debe seguir un dispatcher con catálogo y argumentos validados. Un router equivocado puede producir una llamada innecesaria aunque los argumentos sean válidos.

## 3. Arquitectura

```mermaid
flowchart TD
    A[Solicitud] --> B[Router]
    B --> C{Herramienta permitida}
    C -->|calculadora| D[Parser aritmetico]
    C -->|clima| E[Mock local]
    C -->|producto| F[Catalogo mock]
    C -->|documentos| G[FAQ local]
    C -->|desconocida| H[Rechazo]
    D --> I[Resultado tipado]
    E --> I
    F --> I
    G --> I
```

## 4. Preparación del entorno

Usamos expresiones regulares, pandas y el registro Pydantic de herramientas. Las funciones no dependen de APIs externas; la batería se resuelve localmente.

In [ ]:
import re

import pandas as pd

from llm_engineering.environment import preparar_entorno
from llm_engineering.evaluation.metrics import classification_accuracy
from llm_engineering.tools import ToolName, execute_tool

raiz_curso = preparar_entorno()

## 5. Definición del problema

Enrutaremos solicitudes a calculadora, clima simulado, búsqueda de productos, búsqueda documental o respuesta directa. El conjunto de cinco ejemplos está etiquetado manualmente e incluye una consulta que no debe llamar herramientas.

## 6. Línea base

Una regla de prioridad explícita sirve como línea base y permite inspeccionar por qué se eligió cada herramienta. Extraemos operaciones aritméticas solo en solicitudes que claramente piden calcular.

In [ ]:
def normalizar(texto: str) -> str:
    """Convierte texto a minúsculas sin tildes para reglas simples."""
    return (texto.casefold().replace('á', 'a').replace('é', 'e').replace('í', 'i')
            .replace('ó', 'o').replace('ú', 'u'))

def enrutar_solicitud(texto: str) -> dict[str, object]:
    """Selecciona una herramienta por reglas y construye sus argumentos."""
    consulta = normalizar(texto)
    operacion = re.search(r'[-+]?\d+(?:\.\d+)?(?:\s*[-+*/]\s*[-+]?\d+(?:\.\d+)?)+', texto)
    if any(palabra in consulta for palabra in ('calcula', 'calcular', 'suma')) and operacion:
        return {'tool_name': ToolName.CALCULATOR.value, 'arguments': {'expression': operacion.group(0)}}
    if 'clima' in consulta or 'temperatura' in consulta:
        ciudad = 'Santiago' if 'santiago' in consulta else 'Valparaiso'
        return {'tool_name': ToolName.WEATHER.value, 'arguments': {'city': ciudad}}
    if 'producto' in consulta or 'mochila' in consulta or 'botella' in consulta:
        return {'tool_name': ToolName.PRODUCT_SEARCH.value, 'arguments': {'query': texto}}
    if any(palabra in consulta for palabra in ('envio', 'paquete', 'devolucion', 'reembolso', 'pedido')):
        return {'tool_name': ToolName.DOCUMENT_SEARCH.value, 'arguments': {'query': texto}}
    return {'tool_name': 'respuesta_directa', 'arguments': {}}

print(enrutar_solicitud('calcula 12 * 3'))

## 7. Implementación

El router propone un nombre y argumentos. Solo los nombres presentes en el registro se ejecutan; `respuesta_directa` es una ruta de control, no una herramienta. Validación Pydantic ocurre dentro del dispatcher antes de llamar una función.

In [ ]:
def despachar(texto: str) -> dict[str, object]:
    """Enruta y ejecuta solo nombres reconocidos en el registro."""
    decision = enrutar_solicitud(texto)
    nombre = str(decision['tool_name'])
    if nombre == 'respuesta_directa':
        return {**decision, 'success': True, 'output': 'No se requiere herramienta.', 'error': None}
    resultado = execute_tool(nombre, decision['arguments'])
    return {**decision, **resultado.model_dump()}

print(despachar('busca mochila'))

## 8. Experimentos

Aplicamos el mismo router a solicitudes etiquetadas. Cambia la entrada, no el catálogo. Registramos tanto selección como éxito del dispatcher para localizar si el fallo es de intención o de argumentos.

In [ ]:
pruebas = [
    {'solicitud': 'calcula 12 * 3', 'esperada': 'calculadora'},
    {'solicitud': 'temperatura de Santiago', 'esperada': 'clima_mock'},
    {'solicitud': 'busca una mochila', 'esperada': 'busqueda_productos'},
    {'solicitud': 'donde esta mi paquete', 'esperada': 'busqueda_documentos'},
    {'solicitud': 'hola', 'esperada': 'respuesta_directa'},
]
filas = []
for prueba in pruebas:
    resultado = despachar(prueba['solicitud'])
    filas.append({
        'solicitud': prueba['solicitud'],
        'esperada': prueba['esperada'],
        'seleccionada': resultado['tool_name'],
        'argumentos_validos': resultado['success'],
        'error': resultado['error'],
    })
tabla_router = pd.DataFrame(filas)
tabla_router['acierto_router'] = tabla_router['esperada'] == tabla_router['seleccionada']
display(tabla_router)
tabla_router.to_csv(raiz_curso / 'outputs' / '08_02_router_results.csv', index=False)

## 9. Evaluación

Accuracy mide selección de ruta; la tasa de éxito del dispatcher mide herramientas que aceptaron argumentos y completaron. Reportamos fallos por fila para distinguir un router incorrecto de una herramienta rota.

In [ ]:
metricas_router = pd.DataFrame([{
    'tool_selection_accuracy': classification_accuracy(
        tabla_router['seleccionada'].tolist(), tabla_router['esperada'].tolist()
    ),
    'argument_validity_rate': float(tabla_router['argumentos_validos'].mean()),
    'failure_rate': float((~tabla_router['argumentos_validos']).mean()),
    'llamadas_llm': 0,
}])
display(metricas_router)
metricas_router.to_csv(raiz_curso / 'outputs' / '08_02_router_metrics.csv', index=False)

## 10. Discusión

Las reglas son transparentes y baratas, pero frágiles ante paráfrasis y solapamiento de intención. Un clasificador LLM puede ampliar cobertura, aunque añade latencia y fallos. En ambos casos, el dispatcher debe volver a validar nombre y argumentos y aplicar permisos independientes del router.

## 11. Ejercicios

1. Añade solicitudes ambiguas y define una ruta de aclaración.
2. Cambia el orden de las reglas y analiza conflictos entre intenciones.
3. Mide accuracy por nombre de herramienta.
4. Añade una decisión desconocida y verifica que el dispatcher la rechace.

## 12. Desafío

Compara este router con selección de herramientas de un modelo local sobre 50 solicitudes etiquetadas. Limita la prueba a una invocación por solicitud, reporta exactitud, fallos, latencia y llamadas, y conserva un conjunto de prueba aparte.

## 13. Ideas clave

- Enrutamiento y ejecución son etapas distintas.
- Las salidas del router siempre pasan por el registro y los esquemas Pydantic.
- Mide tool-selection accuracy y failure rate por separado.
- La ruta directa y la abstención son resultados válidos.